In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torchvision import transforms
from google.colab import files
import os

# ==============================================================================
# STEP 1: UPLOAD YOUR OWN REAL FRUIT IMAGES
# ==============================================================================
print("Please select and upload 3 to 5 fruit images (Apples/Bananas) from your computer:")
uploaded = files.upload()

# Get the filenames of uploaded images
image_paths = list(uploaded.keys())
print(f"\nSuccessfully uploaded {len(image_paths)} image(s): {image_paths}")

# ==============================================================================
# STEP 2: DEFINE PREPROCESSING & AUGMENTATION PIPELINES
# ==============================================================================

def preprocess_image(image_path):
    """
    General Image Preprocessing Steps:
    1. Read Image using OpenCV
    2. Convert BGR color space to standard RGB
    3. Resize to standard dimensions (128x128 pixels)
    4. Apply Gaussian Blur to remove noise
    5. Normalize pixel values from [0, 255] to [0.0, 1.0]
    """

    # Load image file
    raw_bgr = cv2.imread(image_path)

    # A. Color Conversion: BGR -> RGB
    raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)

    # B. Uniform Resizing
    resized_rgb = cv2.resize(
        raw_rgb,
        (128, 128),
        interpolation=cv2.INTER_AREA
    )

    # C. Noise Reduction: Gaussian Blur filter
    denoised_rgb = cv2.GaussianBlur(
        resized_rgb,
        (3, 3),
        0
    )

    # D. Pixel Value Normalization
    normalized_float = denoised_rgb.astype(np.float32) / 255.0

    return raw_rgb, normalized_float


# Data Augmentation Pipeline
augmentation_pipeline = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(degrees=30),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3),
    transforms.ToTensor()
])

# ==============================================================================
# STEP 3: PROCESS AND DISPLAY BEFORE VS. AFTER
# ==============================================================================

for img_path in image_paths:

    # Preprocess uploaded image
    raw_original, preprocessed_img = preprocess_image(img_path)

    # Generate augmented variation
    img_uint8 = (preprocessed_img * 255).astype(np.uint8)

    augmented_tensor = augmentation_pipeline(img_uint8)

    augmented_img = augmented_tensor.permute(
        1, 2, 0
    ).numpy()

    # Display 3 stages side-by-side
    fig, axes = plt.subplots(
        1, 3,
        figsize=(12, 4)
    )

    # 1. Raw Original Input
    axes[0].imshow(raw_original)
    axes[0].set_title(
        f"1. Raw Uploaded Input\n"
        f"({os.path.basename(img_path)})"
    )
    axes[0].axis("off")

    # 2. Preprocessed Output
    axes[1].imshow(preprocessed_img)
    axes[1].set_title(
        "2. Preprocessed Image\n"
        "(128x128 Resized, Denoised, Normalized)"
    )
    axes[1].axis("off")

    # 3. Augmented Output
    axes[2].imshow(augmented_img)
    axes[2].set_title(
        "3. Data Augmented Output\n"
        "(Rotated/Flipped for Dataset Expansion)"
    )
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()